# M3 step 12: the decode, detect and stamp pipeline (tasks 7 and 10)

This joins everything into the pipeline from the proposal (`veristamp_pipeline.py`, which the demo can
import):

1. Read the stamp with `extract`. If it is valid, return its verdict and skip the detectors.
2. If not, run M1 and M2, combine them with the fusion model, and apply the Uncertain thresholds.
3. Stamp the final verdict with `embed` and return the stamped photo.

The notebook runs the pipeline on Real and AI GenImage test photos and measures:

- **AI photos stamped as Real**, the system metric from the proposal, and the other verdict counts;
- **false skips:** unstamped photos whose random pixels pass the checksum, so the detectors would be
  skipped by mistake;
- what happens when a pipeline output comes back: unchanged (PNG), after JPEG 90, and, if a GPU is
  available, after a VAE round trip, where the stamp should break and the detectors run again.

With the dummy detector this tests the plumbing only (task 7). Switch the real detectors on below
when they are ready (task 10).

**GPU:** not needed, except for the optional VAE check.

## Setup

**Kaggle:** turn on *Settings > Accelerator > GPU T4 x2* (only notebooks 8, 9 and 11 need it) and
*Internet on*, and attach the `veristamp-m3-inputs` dataset (README.md, "Running on Kaggle"). The first
cell then finds the code, installs what is missing and moves into `m3_experiments`. No path to edit.

**Colab or a local machine:** set the paths once in `settings.py`, then run from the `m3_experiments`
folder.

In [1]:
# Kaggle setup: find the dataset wherever Kaggle put it, copy the code, point the notebooks at the files.
import os, shutil, subprocess, sys
from pathlib import Path

def find(rel, max_depth=6):
    for depth in range(max_depth + 1):
        hits = sorted(Path('/kaggle/input').glob('/'.join(['*'] * depth + [rel])))
        if hits:
            return hits[0]
    return None

if Path('/kaggle/input').is_dir() and not Path('settings.py').exists():
    code = find('m3_experiments/settings.py')
    model = find('m3/best.pt')
    print('Code found at: ', code)
    print('Model found at:', model)
    if code is None or model is None:
        print(subprocess.run(['find', '/kaggle/input', '-maxdepth', '4'], capture_output=True, text=True).stdout[:3000])
        raise SystemExit('The veristamp-m3-inputs dataset is not attached, or has a different layout (see the list above).')
    repo = Path('/kaggle/working/VeriStamp')
    if not (repo / 'm3_experiments' / 'settings.py').exists():
        shutil.rmtree(repo, ignore_errors=True)          # removes a failed earlier clone
        shutil.copytree(code.parent.parent, repo, ignore=shutil.ignore_patterns('.git'))
    # Each item is found on its own, so the inputs can come in several zips or datasets.
    paths = {'M3E_MODEL_PATH': 'm3/best.pt', 'M3E_TEST_DIR': 'm3/test', 'M3E_TEST_MANIFEST': 'm3/test_manifest.csv',
             'M3E_GENIMAGE_SPLIT_DIR': 'genimage/splits', 'M3E_GENIMAGE_ROOT': 'genimage/images',
             'M3E_M1_CHECKPOINT': 'm1/best.pt', 'M3E_M1_VSLIB_DIR': 'm1/vslib_m1_final'}
    for name, rel in paths.items():
        hit = find(rel)
        os.environ[name] = str(hit if hit else model.parent.parent / rel)   # the notebooks run from here inherit these
        print(f'  {rel:<22} {"found" if hit else "not in the inputs"}')
    os.environ['M3E_WORK_DIR'] = '/kaggle/working/m3_work'
    os.chdir(repo / 'm3_experiments')
    for pkg in ('diffusers', 'lpips'):
        try:
            __import__(pkg)
        except ImportError:
            subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pkg], check=True)
print('Working in', Path.cwd())

Working in /kaggle/working/VeriStamp/m3_experiments


In [2]:
# On Colab instead, uncomment and run once per session:
# from google.colab import drive; drive.mount('/content/drive')
# !git clone https://github.com/KruthikaVirupakshappa/VeriStamp.git /content/VeriStamp
# %cd /content/VeriStamp/m3_experiments
# !pip install -q diffusers transformers accelerate scikit-image lpips

import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))
import numpy as np
import pandas as pd
import torch
from PIL import Image
import matplotlib.pyplot as plt

import settings as S
import m3_eval as E
mw = S.setup()                      # imports ../m3_watermark/m3_watermark.py as it is
torch.manual_seed(S.SEED)
np.random.seed(S.SEED)

m3 = mw.load_m3(S.MODEL_PATH)       # load once, reuse for every photo
DEVICE = m3.device
print('Model epoch', m3.epoch, 'on', DEVICE)

Model epoch 97 on cuda


## Detectors

In [3]:
from detectors import DummyDetector, M1Detector, M2Detector, DetectorSet, late_fusion

# Switch each detector on when it is ready. With everything False, the dummy detector stands in for
# all of them, so the notebook runs end to end now (task 7) and only these switches change later.
# kaggle_run_all.ipynb sets these through environment variables; otherwise write True or False here.
import os
USE_M1 = os.environ.get('M3E_USE_M1', '0') == '1'           # M1 is trained (README_M1.md)
USE_M2 = os.environ.get('M3E_USE_M2', '0') == '1'           # waiting for Supriya
USE_FUSION = os.environ.get('M3E_USE_FUSION', '0') == '1'   # late fusion (average) until a trained fusion model exists

detectors = {}
if USE_M1:
    detectors['m1_resnet'] = M1Detector(S.M1_VSLIB_DIR, S.M1_CHECKPOINT)
if USE_M2:
    detectors['m2_clip'] = M2Detector()
if not detectors:
    detectors['dummy'] = DummyDetector()
detector_set = DetectorSet(detectors, fusion=late_fusion if USE_FUSION else None)
DECIDE_WITH = 'fusion' if 'fusion' in detector_set.names else detector_set.names[0]
print('Detectors:', detector_set.names, '| verdict made by:', DECIDE_WITH)
if 'dummy' in detectors:
    print('DUMMY DETECTOR: these numbers test the code only. Do not report them.')

Detectors: ['m1_resnet'] | verdict made by: m1_resnet


In [4]:
from veristamp_pipeline import VeriStampPipeline
lo, hi = S.THRESHOLDS.get(DECIDE_WITH, (0.35, 0.65))
pipeline = VeriStampPipeline(mw, m3, detector_set, decide_with=DECIDE_WITH, t_low=lo, t_high=hi)
print('Thresholds:', lo, hi, '(placeholders until the team fits them on validation)')

Thresholds: 0.35 0.65 (placeholders until the team fits them on validation)


## Photos

In [5]:
N_PER_CLASS = 200
sample = E.load_genimage_test(S.GENIMAGE_SPLIT_DIR, S.GENIMAGE_ROOT, N_PER_CLASS, S.SEED, min_side=224)
sample['class'] = sample['label'].map({0: 'Real', 1: 'AI'})
TEST_VAE = torch.cuda.is_available()      # the optional VAE pass needs a GPU to be quick
vae = E.load_vae(DEVICE, S.VAE_NAME) if TEST_VAE else None

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_validators.py:206: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


## Run the pipeline, then send its output back in

In [6]:
rows = []
for _, s in E.tqdm(sample.iterrows(), total=len(sample)):
    image_id = str(s['image_id'])
    photo = E.save_as(E.open_rgb(s['path']), 'PNG')
    first = pipeline.run(photo, image_id)
    returns = {'returned as PNG': E.save_as(first['photo'], 'PNG'),
               'returned after JPEG 90': E.save_as(first['photo'], 'JPEG', 90)}
    if vae is not None:
        returns['returned after VAE round trip'] = E.vae_roundtrip(vae, first['photo'])
    row = {'image_id': image_id, 'class': s['class'], 'generator': s['generator'],
           'first_source': first['source'], 'first_verdict': first['verdict'], 'first_p_ai': first['p_ai'],
           'seconds': first['seconds']}
    for name, im in returns.items():
        again = pipeline.run(im, image_id)
        row[f'{name}: source'] = again['source']
        row[f'{name}: same verdict'] = again['verdict'] == first['verdict']
    rows.append(row)
runs = pd.DataFrame(rows)

  0%|          | 0/400 [00:00<?, ?it/s]

## System metrics

In [7]:
fresh = runs[runs.first_source == 'detectors']
counts = pd.crosstab(fresh['class'], fresh['first_verdict']).reindex(columns=['Real', 'Uncertain', 'AI'], fill_value=0)
n_ai = int((fresh['class'] == 'AI').sum())
ai_as_real = int(counts.loc['AI', 'Real']) if 'AI' in counts.index else 0
system = {
    'photos': len(runs),
    'false skips (unstamped photo read as stamped)': int((runs.first_source == 'stamp').sum()),
    'AI photos stamped as Real': f'{ai_as_real} of {n_ai} ({ai_as_real / max(n_ai, 1):.1%})',
    'Real photos stamped as AI': int(counts.loc['Real', 'AI']) if 'Real' in counts.index else 0,
    'Uncertain share': f"{(fresh.first_verdict == 'Uncertain').mean():.1%}",
    'mean seconds per photo (first pass)': round(runs.seconds.mean(), 3),
}
for col in [c for c in runs if c.endswith(': source')]:
    name = col[:-len(': source')]
    from_stamp = runs[col] == 'stamp'
    system[f'{name}: answered from stamp'] = f'{from_stamp.mean():.1%}'
    system[f'{name}: same verdict as first pass'] = f"{runs[f'{name}: same verdict'].mean():.1%}"
display(counts)
pd.Series(system, name='value').to_frame()

first_verdict,Real,Uncertain,AI
class,,,
AI,93,15,92
Real,175,12,13


,value
photos,400
false skips (unstamped photo read as stamped),0
AI photos stamped as Real,93 of 200 (46.5%)
Real photos stamped as AI,13
Uncertain share,6.8%
mean seconds per photo (first pass),0.029
returned as PNG: answered from stamp,93.5%
returned as PNG: same verdict as first pass,99.5%
returned after JPEG 90: answered from stamp,88.5%
returned after JPEG 90: same verdict as first pass,98.8%


## Save

In [8]:
stem = 'pipeline' + ('_DUMMY' if 'dummy' in detectors else '')
runs.to_csv(S.RESULTS_DIR / f'{stem}_per_photo.csv', index=False)
table = pd.Series(system, name='value').rename_axis('metric').reset_index()
table.to_csv(S.RESULTS_DIR / f'{stem}_summary.csv', index=False)
(S.RESULTS_DIR / f'{stem}_summary.md').write_text(E.to_markdown(table) + '\n\n' + E.to_markdown(counts.reset_index()) + '\n')
E.save_results(S.RESULTS_DIR, stem, meta={'detectors': detector_set.names, 'decide_with': DECIDE_WITH,
               'thresholds': [lo, hi], 'n_per_class': N_PER_CLASS, 'environment': E.environment_info()})
print(E.to_markdown(table))

| metric | value |
|---|---|
| photos | 400 |
| false skips (unstamped photo read as stamped) | 0 |
| AI photos stamped as Real | 93 of 200 (46.5%) |
| Real photos stamped as AI | 13 |
| Uncertain share | 6.8% |
| mean seconds per photo (first pass) | 0.0290 |
| returned as PNG: answered from stamp | 93.5% |
| returned as PNG: same verdict as first pass | 99.5% |
| returned after JPEG 90: answered from stamp | 88.5% |
| returned after JPEG 90: same verdict as first pass | 98.8% |
| returned after VAE round trip: answered from stamp | 17.8% |
| returned after VAE round trip: same verdict as first pass | 82.0% |
